# SatQuery Phase 3A — Kaggle grounding baseline
This notebook only provisions Kaggle and calls repository entrypoints. Enable a GPU and Internet before running all cells.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', 'torch==2.8.0', 'torchvision==0.23.0', 'torchaudio==2.8.0', '--index-url', 'https://download.pytorch.org/whl/cu126'], check=True)

In [ ]:
import os, platform, subprocess, sys
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Enable a GPU in Kaggle Settings.'
gpu = torch.cuda.get_device_properties(0)
print({'python': platform.python_version(), 'pytorch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': gpu.name, 'vram_gib': round(gpu.total_memory / 1024**3, 2)})

In [ ]:
REPO_URL = os.environ.get('SATQUERY_REPO_URL', 'https://github.com/bishuk-dev/SIH-26167-SATQuery.git')
REPO_DIR = Path('/kaggle/working/SIH-26167-SATQuery')
OUTPUT_DIR = Path('/kaggle/working/satquery-output/phase3a-grounding-dino')
DATA_ROOT = Path(os.environ.get('SATQUERY_VRSBENCH_ROOT', '/kaggle/working/vrsbench-grounding'))
if (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', f'{REPO_DIR}[grounding]'], check=True)

In [ ]:
subprocess.run([sys.executable, '-m', 'ml.evaluation.prepare_vrsbench_grounding', '--data-root', str(DATA_ROOT), '--allow-download'], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, '-m', 'ml.evaluation.run_phase3a_grounding', '--data-root', str(DATA_ROOT), '--output-dir', str(OUTPUT_DIR), '--split', 'validation', '--device', 'cuda', '--allow-download'], cwd=REPO_DIR, check=True)

In [ ]:
print('Phase 3A artifacts:')
for path in sorted(OUTPUT_DIR.rglob('*')):
    if path.is_file(): print(path.relative_to(OUTPUT_DIR), path.stat().st_size)